^C


In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd
import tensorflow as keras
from keras import layers, Input, Model, ops
from keras.layers import Embedding, Dense
import tensorflow as tf
from collections import defaultdict
from math import radians, sin, cos, sqrt, atan2
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler



In [2]:
def HaversineMiles(coord1, coord2):
    R = 3958.8  # Earth radius in miles
    lat1, lon1 = map(radians, coord1)
    lat2, lon2 = map(radians, coord2)
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    
    return R * c

def getProximityBonus(
    hider_lat, hider_long,
    seeker_lats, seeker_longs,
    seeker_start_lats, seeker_start_longs,
    max_movement_bonus=0.35):
    hider_pos = (hider_lat, hider_long)

    movement_bonuses = []
    for start_lat, start_long, end_lat, end_long in zip(seeker_start_lats, seeker_start_longs, seeker_lats, seeker_longs):
        start_dist = HaversineMiles((start_lat, start_long), hider_pos)
        end_dist = HaversineMiles((end_lat, end_long), hider_pos)
        delta = start_dist - end_dist

        if delta <= 0 or start_dist == 0:
            movement_bonuses.append(0.0)
        else:
            movement_ratio = delta / start_dist
            movement_bonuses.append(movement_ratio * max_movement_bonus)

    return sum(movement_bonuses)



def BuildEdgeIDHash(self):
    return dict(zip(self['edge_id'], zip(self['u'], self['v'])))

def BuildNodeToEdgesHash(self):
    node_to_edges = defaultdict(list)

    for _, row in self.iterrows():
        u = row['u']
        edge_id = row['edge_id']
        node_to_edges[u].append(edge_id)
    return node_to_edges

def BuildStaticFeatures(self):
    return self[['Distance_miles', 'TimeToCross_minutes', 'lon_u',
                        'lat_u', 'isHighPriority_u', 'lon_v', 'lat_v', 
                        'isHighPriority_v','heading_u', 'heading_v']].values

class PolicyHeader(tf.keras.Model):
    def __init__(self, num_edges, embed_dim):
        super().__init__()
        self.dense1 = Dense(128, activation='relu')
        self.dense2 = Dense(128, activation='relu')
        self.agent_proj = Dense(embed_dim)

        self.edge_embeddings = Embedding(num_edges, embed_dim)
        self.static_proj = Dense(embed_dim)

    def call(self, local_obs_batch, valid_edge_ids_batch, edge_static_features):
        if len(local_obs_batch.shape) == 3:
            local_obs_batch = tf.squeeze(local_obs_batch, axis=1)
        #print(local_obs_batch.shape)
        x = self.dense1(local_obs_batch)    
        x = self.dense2(x)                   
        agent_features = self.agent_proj(x)  

        all_probs = []

        batch_size = tf.shape(agent_features)[0]

        for i in range(5):
            ids = valid_edge_ids_batch[i] 

            valid_edge_vecs = self.edge_embeddings(ids)  
            valid_edge_static = tf.gather(edge_static_features, ids)
            #print(valid_edge_vecs)
            static_proj = self.static_proj(valid_edge_static)  

            final_edge_vecs = valid_edge_vecs + static_proj  

            seeker_vec = agent_features[i]  
            seeker_vec = tf.expand_dims(seeker_vec, axis=0)  

            # Dot: (1, embed_dim) * (num_valid, embed_dim) → (num_valid,)
            scores = tf.reduce_sum(seeker_vec * final_edge_vecs, axis=-1)
            #print(scores)
            probs = tf.nn.softmax(scores) 
            #print(probs)
            all_probs.append(probs)

        return all_probs

class ValueHeader(tf.keras.Model):
    def __init__(self, num_seekers=5):
        super().__init__()
        input_dim = 7 + num_seekers * 6
        
        self.dense1 = layers.Dense(256, activation='relu')
        self.dropout1 = layers.Dropout(0.3)
        
        self.dense2 = layers.Dense(256, activation='relu')
        self.dropout2 = layers.Dropout(0.3)
        
        self.dense3 = layers.Dense(256, activation='relu')
        self.dropout3 = layers.Dropout(0.2)
        
        self.dense4 = layers.Dense(256, activation='relu')
        self.dropout4 = layers.Dropout(0.2)
        
        self.output_layer = layers.Dense(1, activation='tanh')

    def call(self, inputs, training=False):
        x = self.dense1(inputs)
        x = self.dropout1(x, training=training)
        
        x = self.dense2(x)
        x = self.dropout2(x, training=training)
        
        x = self.dense3(x)
        x = self.dropout3(x, training=training)
        
        x = self.dense4(x)
        x = self.dropout4(x, training=training)
        
        value = self.output_layer(x)
        return value

In [3]:
nodes = pd.read_csv(r"C:\Users\asriv\Project Amber\DSMMetro_nodes.csv")
nodes.set_index('Node', inplace=True)
nodes = nodes.drop('Unnamed: 0', axis=1)

nodes.head()

,longitude,latitude,isHighPriority
Node,,,
IA158939275,-92.858306,40.783423,False
IA158943282,-92.925092,40.739831,False
IA158948478,-92.695568,40.739086,False
IA158948866,-92.695845,40.775949,False
IA158948924,-92.657867,40.841815,False


In [4]:
df = pd.read_parquet(r"C:\Users\asriv\source\repos\Nemesis-Engine\Nemesis-Engine\replays")
seeker_pos = df['seeker_positions'].tolist()
starting_seeker_pos = df['starting_seeker_positions'].tolist()

seeker_start_lats = [[float(nodes.loc[i][['latitude']].values[0]) for i in j] for j in starting_seeker_pos]
seeker_start_longs = [[float(nodes.loc[i][['longitude']].values[0]) for i in j] for j in starting_seeker_pos]
seeker_lats = [[float(nodes.loc[i][['latitude']].values[0]) for i in j] for j in seeker_pos]
seeker_longs = [[float(nodes.loc[i][['longitude']].values[0]) for i in j] for j in seeker_pos]
hider_lat = [float(nodes.loc[i][['latitude']].values[0]) for i in df['hider_position']]
hider_long = [float(nodes.loc[i][['longitude']].values[0]) for i in df['hider_position']]

proximity_bonus = [
    getProximityBonus(h_lat, h_long, s_lats, s_longs, ss_lats, ss_longs)
    for h_lat, h_long, s_lats, s_longs, ss_lats, ss_longs in zip(hider_lat, hider_long, seeker_lats, seeker_longs,seeker_start_lats, seeker_start_longs)
]


In [5]:
df['ProximityBonus'] = [i if j == -1 else 0 for i,j in zip(proximity_bonus, df['value'])]
df['StateValue'] = df['value'] + df['ProximityBonus']

df

,state_vector,observation_vectors,learned_policy,starting_seeker_positions,seeker_positions,hider_position,value,ProximityBonus,StateValue
0,"[[0.0, 0.2800215800825135, 41.503765213540404,...","[[0.0, 0.0, 0.6301773, 0.27882454, 42.9382, 41...","[{""12927"": 0.40923076923076923, ""12928"": 0.060...","[IA9674819555, IA160738237, IA160852521, IA160...","[IA9674819555, IA160738237, IA160852521, IA160...",IA160123609,1.0,0.0,1.0
1,"[[1.0, 0.2800215800825135, 41.503765213540404,...","[[0.0, 0.0, 0.6301773, 0.27882454, 42.9382, 41...","[{""12927"": 0.2, ""12928"": 0.2, ""12929"": 0.2, ""2...","[IA9674819555, IA160738237, IA160852521, IA160...","[IA9674819555, IA160738237, IA160852521, IA160...",IA160123609,1.0,0.0,1.0
2,"[[2.0, 0.2800215800825135, 41.51556817809155, ...","[[0.0, 0.0, 0.6301773, 0.27882454, 42.9382, 41...","[{""12930"": 0.7018796992481203, ""27126"": 0.2112...","[IA9674819555, IA160738237, IA160852521, IA160...","[IA9674819564, IA160789440, IA8255644181, IA16...",IA160123609,1.0,0.0,1.0
3,"[[3.0, 0.2800215800825135, 41.51556817809155, ...","[[0.0, 0.0, 0.6301773, 0.27882454, 42.9382, 41...","[{""12930"": 0.3333333333333333, ""27126"": 0.3333...","[IA9674819555, IA160738237, IA160852521, IA160...","[IA9674819564, IA160789440, IA8255644181, IA16...",IA160123609,1.0,0.0,1.0
4,"[[4.0, 0.2800215800825135, 42.76543239255993, ...","[[0.0, 0.0, 0.6301773, 0.27882454, 42.9382, 41...","[{""12158"": 0.11012094207511139, ""12159"": 0.114...","[IA9674819555, IA160738237, IA160852521, IA160...","[IA2320013230, IA160738237, IA8255644180, IA39...",IA160123609,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...
47495,"[[63.0, 0.02130130427388938, 10.39515074916416...","[[0.0, 4.0, 0.48431098, 0.0293489, 8.926157, 1...","[{""6497"": 0.125, ""6498"": 0.125, ""6499"": 0.125,...","[IA160733934, IA160938758, IA5377365202, IA121...","[IA160848532, IA633038363, IA989054153, IA1608...",IA2023806014,1.0,0.0,1.0
47496,"[[64.0, 0.02130130427388938, 11.07377107329440...","[[0.0, 4.0, 0.48431098, 0.0293489, 8.926157, 1...","[{""8791"": 0.7944656488549618, ""8792"": 0.047709...","[IA160733934, IA160938758, IA5377365202, IA121...","[IA5838882132, IA160781614, IA160942310, IA160...",IA2023806014,1.0,0.0,1.0
47497,"[[65.0, 0.02130130427388938, 11.07377107329440...","[[0.0, 4.0, 0.48431098, 0.0293489, 8.926157, 1...","[{""8791"": 0.2, ""8792"": 0.2, ""23802"": 0.2, ""242...","[IA160733934, IA160938758, IA5377365202, IA121...","[IA5838882132, IA160781614, IA160942310, IA160...",IA2023806014,1.0,0.0,1.0
47498,"[[66.0, 0.02130130427388938, 10.39515074916416...","[[0.0, 4.0, 0.48431098, 0.0293489, 8.926157, 1...","[{""6497"": 0.032288880397401604, ""6498"": 0.0309...","[IA160733934, IA160938758, IA5377365202, IA121...","[IA160848532, IA633038363, IA160942275, IA9526...",IA2023806014,1.0,0.0,1.0


In [6]:
df.groupby('value').count()

,state_vector,observation_vectors,learned_policy,starting_seeker_positions,seeker_positions,hider_position,ProximityBonus,StateValue
value,,,,,,,,
-1.0,34920,34920,34920,34920,34920,34920,34920,34920
1.0,12580,12580,12580,12580,12580,12580,12580,12580


In [7]:
# edges = pd.read_csv(r'C:\Users\asriv\source\repos\Nemesis-Engine\Nemesis-Engine\data\DSMMetro_edges.csv')

# EdgeIDHash = BuildEdgeIDHash(edges)
# NodeToEdgesHash = BuildNodeToEdgesHash(edges)
# # staticFeatures = BuildStaticFeatures(edges)
# results = []

# for pos in seeker_pos:
#     results.append([NodeToEdgesHash[i] for i in pos])

In [8]:
states = df['state_vector'].values
states = np.array([i[0] for i in states])

values = df['StateValue'].values
#values

In [9]:
# Hyperparameters
num_seekers = 5
input_dim = 7 + num_seekers * 4
batch_size = 32
epochs = 15

# Dummy data
X = states
y = values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1141)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = ValueHeader(num_seekers=num_seekers)
loss_fn = tf.keras.losses.MeanSquaredError()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# Additional metrics
mae_fn = tf.keras.losses.MeanAbsoluteError()

# Dataset creation
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1024).batch(batch_size)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(batch_size)

# Model, loss, optimizer
#model = ValueHeader(num_seekers=num_seekers)  # Assuming this is defined
loss_fn = tf.keras.losses.MeanSquaredError()
mae_fn = tf.keras.losses.MeanAbsoluteError()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# Training loop
for epoch in range(epochs):
    epoch_mse = 0.0
    epoch_mae = 0.0
    all_y_true = []
    all_y_pred = []

    progress_bar = tqdm(enumerate(train_dataset), total=len(X_train) // batch_size, desc=f"Epoch {epoch+1}")

    for step, (x_batch, y_batch) in progress_bar:
        with tf.GradientTape() as tape:
            predictions = model(x_batch, training=True)
            loss = loss_fn(y_batch, predictions)
        grads = tape.gradient(loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))

        epoch_mse += loss.numpy()
        epoch_mae += mae_fn(y_batch, predictions).numpy()

    # Evaluation on test set
    all_y_true = []
    all_y_pred = []
    for x_batch, y_batch in test_dataset:
        predictions = model(x_batch, training=False)
        all_y_true.extend(y_batch.numpy().flatten())
        all_y_pred.extend(predictions.numpy().flatten())

    y_true = np.array(all_y_true)
    y_pred = np.array(all_y_pred)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2_score = 1 - (ss_res / ss_tot)

    avg_mse = epoch_mse / (step + 1)
    avg_mae = epoch_mae / (step + 1)
    print(f"Epoch {epoch + 1} - MSE: {avg_mse:.4f}, MAE: {avg_mae:.4f}, R² (val): {r2_score:.4f}")

Epoch 1: 1188it [00:57, 20.84it/s]                                                                                     


Epoch 1 - MSE: 0.3071, MAE: 0.3298, R² (val): 0.8546


Epoch 2: 1188it [00:55, 21.23it/s]                                                                                     


Epoch 2 - MSE: 0.1218, MAE: 0.1647, R² (val): 0.9416


Epoch 3: 1188it [00:55, 21.30it/s]                                                                                     


...

Epoch 5 - MSE: 0.0478, MAE: 0.0884, R² (val): 0.9714


Epoch 6: 1188it [00:57, 20.72it/s]                                                                                     


Epoch 6 - MSE: 0.0382, MAE: 0.0777, R² (val): 0.9919


Epoch 7:  21%|██████████████▍                                                       | 245/1187 [00:11<00:45, 20.62it/s]


KeyboardInterrupt: 

In [11]:
 #model.save('ValueFunction_batch1.keras')

In [10]:
model.save_weights("ValueFunction_Timelocked_batch0_r2_98.weights.h5")


(17,)

In [62]:
# Hyperparameters
num_seekers = 5
input_dim = 7 + num_seekers * 4
batch_size = 32
epochs = 10

# Dummy data
X_train = (obs_v, 
y_train = pols


# Model, loss, optimizer
model = PolicyHeader(num_edges=len(edges), embed_dim=64)
loss_fn = tf.keras.losses.MeanSquaredError()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# Training loop
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(buffer_size=1024).batch(batch_size)

for epoch in range(epochs):
    epoch_loss = 0.0
    for step, (x_batch, y_batch) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            predictions = model(x_batch)
            loss = loss_fn(y_batch, predictions)
        grads = tape.gradient(loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        epoch_loss += loss.numpy()
    print(f"Epoch {epoch + 1}, Loss: {epoch_loss / (step + 1):.4f}")

NameError: name 'defaultdict' is not defined

In [70]:
X_train.shape

(984, 27)

In [ ]:
e